In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

import os

load_dotenv()

chat = ChatOpenAI(
    base_url = os.getenv("OPENAI_BASE_URL"),
    api_key = os.getenv("OPENAI_API_KEY"), 
    model = os.getenv("OPENAI_MODEL_NAME", "gpt-5.1"),
    temperature = 0.1,
)

In [ ]:
from langchain_core.prompts.few_shot import FewShotChatMessagePromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.callbacks import get_usage_metadata_callback
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache

set_llm_cache(SQLiteCache("cache.db"))

example = [
    {
        "content": "How are you?",
        "result": "잘 지내?"
    },
    {
        "content": "What are you doing?",
        "result": "뭐 하고 있어?"
    },
    {
        "content": "Do you want to grab dinner later?",
        "result": "이따 저녁 먹으러 갈래?"
    },
    {
        "content": "Can you help me with my homework?",
        "result": "숙제 좀 도와줄래?"
    }
]

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "Translate this {content} for me from English to Korean."),
        ("ai", "{result}"),
    ]
)
fewshot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=example
)

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a professional translator. Translate the sentence from English to Korean, keeping the same style as in the examples."),
        fewshot_prompt,
        ("human", "{content}"),
    ]
)

chain = final_prompt | chat

with get_usage_metadata_callback() as usage:
    a = chain.invoke({
        "content": "Call me when you get home."
    })

    b = chain.invoke({
        "content": "Can you help me with my homework?"
    })

print("A:", a.content)
print("A usage:", a.usage_metadata)
print()
print("B:", b.content)
print("B usage:", b.usage_metadata)
print()
print("usage:", usage)